# Preprocessing and NLP Pipeline

This notebook is a **verbatim walkthrough** of the core preprocessing pipeline. It mirrors the production code but is isolated for safe explanation and debugging.

**Source files copied here:**
- nlp/preprocessing.py
- nlp/clause_segmenter.py

The code below is copied exactly as in the project.

## Verbatim Code: nlp/preprocessing.py

In [ ]:
import string
import re

# We use spaCy because it's like a 'linguistic expert' for Python.
# It understands that "running" and "ran" are the same word ("run").
# We disable the parser and NER components since we only need lemmatization —
# this makes processing ~5x faster.
try:
    import spacy
    try:
        # 'sm' stands for 'small' - it's fast and enough for our needs.
        # disable=['parser','ner'] skips expensive components we don't need.
        nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    except Exception:
        nlp = None
except (ImportError, Exception):
    nlp = None


def _clean_raw(text: str) -> str:
    """Steps 1-3: lowercase, strip punctuation, collapse whitespace."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def preprocess_text(text):
    """
    Cleans a single text string so our AI can understand it more easily.
    For large corpora prefer batch_preprocess_texts() which is much faster.
    """
    if not isinstance(text, str):
        return ""

    text = _clean_raw(text)

    # Lemmatization (using spaCy):
    # This turns words like "agreements" into "agreement".
    if nlp:
        try:
            doc = nlp(text)
            # We also ignore 'stopwords' (common words like 'the', 'is' that add no value).
            tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]
            return " ".join(tokens)
        except Exception:
            pass

    # Fallback: If spaCy isn't working, we do a simple split.
    tokens = [t for t in text.split() if len(t) > 2]
    return " ".join(tokens)


def batch_preprocess_texts(texts, batch_size=512):
    """
    Efficiently preprocess a list/Series of texts using spaCy's nlp.pipe(),
    which is significantly faster than calling preprocess_text() row-by-row.
    Falls back to the single-text path if spaCy is unavailable.
    """
    cleaned = [_clean_raw(t) if isinstance(t, str) else "" for t in texts]

    if nlp:
        results = []
        for doc in nlp.pipe(cleaned, batch_size=batch_size):
            tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]
            results.append(" ".join(tokens))
        return results

    # Fallback without spaCy
    return [" ".join(t for t in text.split() if len(t) > 2) for text in cleaned]

if __name__ == "__main__":
    # Test our 'cleaning machine'
    sample = "The parties are currently agreeing to the terms!"
    print(f"Before: {sample}")
    print(f"After : {preprocess_text(sample)}")

### What this does, line by line
- Imports: `string` and `re` support punctuation stripping and whitespace cleanup.
- `try/except` around spaCy ensures the pipeline still works even if spaCy isn't installed.
- `spacy.load(..., disable=[...])` loads only lemmatization for speed.
- `_clean_raw()` uses lowercase + `translate()` + regex to normalize text.
- `preprocess_text()` uses lemmatization when available, or a fallback token filter when not.
- `batch_preprocess_texts()` uses `nlp.pipe()` for fast batching.

### Why this is important
- Legal clauses are noisy; normalization reduces vocabulary size and improves model generalization.
- Lemmatization aligns words with the same meaning ("agreements" → "agreement").
- The fallback path keeps the app resilient on environments like Streamlit Cloud.

### Alternatives considered (and why not used here)
- **NLTK lemmatization**: slower setup and less robust without additional models.
- **Stanza**: higher accuracy but heavier runtime and downloads.
- **Transformer tokenizers** (BERT, etc.): stronger semantic modeling, but far heavier compute for this project's scope and interpretability goals.

## Verbatim Code: nlp/clause_segmenter.py

In [ ]:
import re

def segment_clauses(text):
    """
    Contracts are long! This function splits a long document into smaller 'clauses'.
    Think of it like cutting a long sandwich into bite-sized pieces.
    """
    if not text:
        return []

    # Legal contracts usually use specific symbols to start new sections:
    # 1. Numbers followed by a period (1. , 2. )
    # 2. Words like "Article" or "Section"
    # 3. Double newlines (paragraphs)
    
    # We use 'Regex' (Regular Expressions) which is like a super-powered CTRL+F search.
    # This pattern looks for common legal markers.
    pattern = r'\n\s*\n|(?<=\n)(?=\d+\.\s|Article\s+[IVXLCDM\d]+|SECTION\s+\d+|[A-Z]\.\s)'
    
    # Split the text based on where those markers appear.
    segments = re.split(pattern, text, flags=re.IGNORECASE)
    
    # Final cleanup: Remove whitespace and ignore tiny segments.
    clauses = []
    for seg in segments:
        clean_seg = seg.strip()
        if clean_seg and len(clean_seg) > 10: # Only keep meaningful text
            clauses.append(clean_seg)
            
    return clauses

if __name__ == "__main__":
    test_text = "1. First Clause. \n\n 2. Second Clause. \n Article III: Third Clause."
    print(f"Found {len(segment_clauses(test_text))} clauses.")

### What this does, line by line
- The regex pattern captures common legal section markers and paragraph breaks.
- `re.split(...)` splits on those markers to produce clause segments.
- A length filter removes noise from short fragments.

### Why this is important
- Clause-level analysis is the core unit of risk prediction.
- Splitting into clauses improves interpretability and precision.

### Alternatives considered (and why not used here)
- **PDF layout-based segmentation**: more accurate but depends on PDF parsing, which is noisy.
- **Rule-based parsers per contract type**: more precise but not scalable across document styles.
- **Transformer-based sentence segmentation**: accurate but slower and unnecessary for this stage.